In [ ]:
### SQL Databases

In [1]:
# create a sample SQLite database
import sqlite3
import os
os.makedirs("data/databases", exist_ok=True)

In [4]:
#sample database
conn = sqlite3.connect("data/databases/company.db")
cursor = conn.cursor()

In [8]:
#create table
cursor.execute('''
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    role TEXT NOT NULL,
    salary REAL NOT NULL,
    department TEXT NOT NULL)
''')

In [9]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS projects (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    status TEXT NOT NULL,
    budget REAL NOT NULL,
    lead_id INTEGER)
''')

In [10]:
#Insert sample data
employees =[
    (1, "Alice", "Software Engineer", 90000, "Engineering"),
    (2, "Bob", "Data Scientist", 95000, "Data Science"),
    (3, "Charlie", "Product Manager", 105000, "Product"),
    (4, "David", "UX Designer", 85000, "Design"),
    (5, "Eve", "DevOps Engineer", 92000, "Engineering")
]

projects = [
    (1, "Project Alpha", "In Progress", 500000, 1),
    (2, "Project Beta", "Completed", 300000, 2),
    (3, "Project Gamma", "Not Started", 200000, 3),
    (4, "Project Delta", "In Progress", 400000, 3),
    (5, "Project Epsilon", "Completed", 350000, 1)
]

In [11]:
cursor.executemany('INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?, ?)', employees)
cursor.executemany('INSERT OR REPLACE INTO projects VALUES (?, ?, ?, ?, ?)', projects)

In [12]:
conn.commit()
conn.close()

In [13]:
## Database COntent Extractrion
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

/home/divyanshu/Desktop/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
#Method 1  SQLDATABASE Utility
db= SQLDatabase.from_uri("sqlite:///data/databases/company.db")

#get database info
print(f"Tables: {db.get_table_names()}")
print(f"\nTable DDL:")
print(db.get_table_info())

Tables: ['employees', 'projects']

Table DDL:

CREATE TABLE employees (
	id INTEGER, 
	name TEXT NOT NULL, 
	role TEXT NOT NULL, 
	salary REAL NOT NULL, 
	department TEXT NOT NULL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	salary	department
1	Alice	Software Engineer	90000.0	Engineering
2	Bob	Data Scientist	95000.0	Data Science
3	Charlie	Product Manager	105000.0	Product
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT NOT NULL, 
	status TEXT NOT NULL, 
	budget REAL NOT NULL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	Project Alpha	In Progress	500000.0	1
2	Project Beta	Completed	300000.0	2
3	Project Gamma	Not Started	200000.0	3
*/


/tmp/ipykernel_326456/1008317100.py:5: LangChainDeprecationWarning: The method `SQLDatabase.get_table_names` was deprecated in langchain-community 0.0.1 and will be removed in 1.0. Use `get_usable_table_names` instead.
  print(f"Tables: {db.get_table_names()}")


In [15]:
#Method 2: Custom SQL to Document conversion
from typing import List
from langchain_core.documents import Document
print("\nCustom SQL Processing:")

def sql_to_documents(db_path:str) -> List[Document]:
    """Convert SQL database content into a list of Documents."""
    conn =  sqlite3.connect(db_path)
    cursor = conn.cursor()
    documents =[]

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    for table in tables:
        table_name = table[0]
        
        #Get Table Schema
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()
        column_names = [col[1] for col in columns]

        #Get Table Data
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        #Create Table Overview Document
        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {', '.join(column_names)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"

        # ADD Sample Data
        for row in rows[:5]:  # Include only first 5 rows for brevity
            record = dict(zip(column_names, row))
            table_content += f"Record: {record}\n"
        
        doc = Document(
            page_content=table_content,
            metadata={
                "source": db_path,
                "table": table_name,
                "num_records": len(rows),
                "data_type": "sql_table"
            }
        )
        documents.append(doc)
    conn.close()
    return documents


Custom SQL Processing:


In [16]:
sql_to_documents("data/databases/company.db")

[Document(metadata={'source': 'data/databases/company.db', 'table': 'employees', 'num_records': 5, 'data_type': 'sql_table'}, page_content="Table: employees\nColumns: id, name, role, salary, department\nTotal Records: 5\n\nRecord: {'id': 1, 'name': 'Alice', 'role': 'Software Engineer', 'salary': 90000.0, 'department': 'Engineering'}\nRecord: {'id': 2, 'name': 'Bob', 'role': 'Data Scientist', 'salary': 95000.0, 'department': 'Data Science'}\nRecord: {'id': 3, 'name': 'Charlie', 'role': 'Product Manager', 'salary': 105000.0, 'department': 'Product'}\nRecord: {'id': 4, 'name': 'David', 'role': 'UX Designer', 'salary': 85000.0, 'department': 'Design'}\nRecord: {'id': 5, 'name': 'Eve', 'role': 'DevOps Engineer', 'salary': 92000.0, 'department': 'Engineering'}\n"),
 Document(metadata={'source': 'data/databases/company.db', 'table': 'projects', 'num_records': 5, 'data_type': 'sql_table'}, page_content="Table: projects\nColumns: id, name, status, budget, lead_id\nTotal Records: 5\n\nRecord: {'